# SFT robuste — miniLLM v2.2 → v2.3

Ce notebook reprend le checkpoint
[`bonopassale/miniLLM_v2.2-49M-50200it_3.29Btoks_1.0ep_20261005`](https://huggingface.co/bonopassale/miniLLM_v2.2-49M-50200it_3.29Btoks_1.0ep_20261005)
et refait un SFT conçu pour mieux **extraire les connaissances accumulées au pré-entraînement**
(Wikipédia FR quasi complète + FineWeb-2 FR, ≈3,29 Md tokens), plutôt que de simplement imiter
un style d'instructions génériques.

## Diagnostic rapide (lu dans la fiche du modèle)

- Le SFT v2.2 est composé à **62 % de French Alpaca**, un jeu d'instructions génériques
  (recettes, rédaction, conseils) qui n'est pas ancré sur le corpus de pré-entraînement.
  Le modèle y apprend surtout un *format* de réponse, peu de *rappel factuel*.
- Les seules sources réellement ancrées sur le même corpus (Wikipédia FR) sont PIAF
  (1 881 exemples, 8 %) — aucune donnée FQuAD/SQuAD-FR n'était incluse en v2.2.
  Or PIAF/FQuAD/SQuAD-FR sont justement les jeux de QA extractive construits *sur*
  Wikipédia FR : ils enseignent au modèle le geste même qu'on attend de lui ici — retrouver
  une réponse dans ce qu'il a déjà vu — avec le vocabulaire de son propre pré-entraînement.
- Les données de calcul/logique sont utiles pour le *format* mais un modèle de 49M n'apprendra
  pas à calculer par SFT seul (confirmé par les évals v2.2 : 2/24 en calcul pur) — on les garde
  en faible proportion, sans en attendre de miracle.
- La persona répétée ×10 (5,8 % du SFT v2.1/v2.2) déborde sur d'autres réponses
  (« SQL est développé par DevLab »). On réduit le facteur de répétition.
- `val_loss` **ne prédit pas la qualité** ici (déjà observé par vous : LR plus bas → meilleure
  val_loss mais persona et connaissances dégradées). Ce notebook sélectionne donc le checkpoint
  final sur un **jeu de sondage factuel tenu à l'écart du SFT**, pas seulement sur la loss.

## Architecture (extraite de `config.json` / `inference.py` du dépôt)

| paramètre | valeur |
|---|---|
| paramètres | 48 508 672 (≈32,1M hors embeddings) |
| couches | 10 |
| d_model | 512 |
| têtes d'attention / KV | 8 / 8 (MHA classique, pas de GQA) |
| FFN | SwiGLU, hidden 1408 |
| normalisation | RMSNorm (eps 1e-6) |
| position | RoPE (theta 10000) |
| QK-Norm | oui |
| embeddings | partagés (entrée = sortie) |
| contexte | 512 tokens |
| vocabulaire | 32 000 (BPE byte-level) |

On réutilise cette architecture **telle quelle** (copiée de `inference.py`, avec un seul ajout :
un masque de padding pour l'entraînement par batch, absent du script d'inférence qui travaille
token par token avec KV-cache). Les poids restent 100 % compatibles avec `inference.py` pour le
déploiement final.


In [ ]:
# Installation
!pip -q install -U datasets tokenizers safetensors huggingface_hub accelerate sentencepiece


In [ ]:
# Identifiants — ne JAMAIS coller de token en clair dans une cellule.
# Si tu as déjà collé un token GitHub ou HF dans un chat, considère-le compromis : régénère-le.
import getpass, os

hf_token = getpass.getpass("Token Hugging Face (lecture des datasets privés + push du checkpoint) : ")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)

# Dépôt source et dépôt de destination
SOURCE_REPO = "bonopassale/miniLLM_v2.2-49M-50200it_3.29Btoks_1.0ep_20261005"
DEST_REPO   = "bonopassale/minillm-v2.3-sft"   # adapte le nom si besoin
PUSH_TO_HUB = False  # passe à True quand tu es prêt à publier


In [ ]:
# Téléchargement des fichiers nécessaires du checkpoint source
from huggingface_hub import hf_hub_download
import shutil, os

CKPT_DIR = "/content/base_model"
os.makedirs(CKPT_DIR, exist_ok=True)

for fname in ["config.json", "generation_config.json", "tokenizer.json", "model.safetensors"]:
    path = hf_hub_download(SOURCE_REPO, fname, token=hf_token)
    shutil.copy(path, os.path.join(CKPT_DIR, fname))

print(os.listdir(CKPT_DIR))


## Architecture (copie fidèle + masque de padding pour l'entraînement)

Le `forward` de `Attention` dans `inference.py` ne gère que deux cas : auto-régressif avec
KV-cache (token par token) ou un seul passage causal plein (`is_causal=True`, sans padding, car
l'inférence traite une séquence à la fois). Pour l'entraînement par batch, les exemples ont des
longueurs différentes : il faut un masque combinant **causalité + padding**. C'est le seul
changement apporté ci-dessous ; tout le reste (RMSNorm, RoPE, SwiGLU, QK-Norm, embeddings liés)
est identique à `inference.py`, donc le `state_dict` final est directement chargeable par le
script d'inférence du dépôt.


In [ ]:
import json, math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class ModelConfig:
    vocab_size: int
    n_layers: int
    d_model: int
    n_heads: int
    kv_heads: int
    ffn_hidden: int
    max_seq_len: int
    dropout: float = 0.0
    rope_theta: float = 10000.0
    tie_embeddings: bool = True
    qk_norm: bool = True
    norm_eps: float = 1e-6

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads

    @classmethod
    def from_json(cls, path: str) -> "ModelConfig":
        with open(path, "r", encoding="utf-8") as f:
            d = json.load(f)
        names = {"vocab_size", "n_layers", "d_model", "n_heads", "kv_heads", "ffn_hidden",
                 "max_seq_len", "dropout", "rope_theta", "tie_embeddings", "qk_norm", "norm_eps"}
        return cls(**{k: v for k, v in d.items() if k in names})


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        out = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return out.to(x.dtype) * self.weight.to(x.dtype)


def build_rope_tables(head_dim, max_seq_len, theta, device=None):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32, device=device) / head_dim))
    t = torch.arange(max_seq_len, dtype=torch.float32, device=device)
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()


def apply_rope(x, cos, sin):
    x32 = x.float()
    half = x.shape[-1] // 2
    rot = torch.cat((-x32[..., half:], x32[..., :half]), dim=-1)
    out = x32 * cos[None, :, None, :] + rot * sin[None, :, None, :]
    return out.to(x.dtype)


class Attention(nn.Module):
    # Identique à inference.py, + un chemin d'entraînement par batch avec masque padding+causal.
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_heads, self.kv_heads, self.hd = cfg.n_heads, cfg.kv_heads, cfg.head_dim
        self.n_rep = cfg.n_heads // cfg.kv_heads
        self.dropout = cfg.dropout
        d = cfg.d_model
        self.wq = nn.Linear(d, self.n_heads * self.hd, bias=False)
        self.wk = nn.Linear(d, self.kv_heads * self.hd, bias=False)
        self.wv = nn.Linear(d, self.kv_heads * self.hd, bias=False)
        self.wo = nn.Linear(self.n_heads * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd, cfg.norm_eps) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd, cfg.norm_eps) if cfg.qk_norm else None

    def forward(self, x, cos, sin, attn_bias: Optional[torch.Tensor] = None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.hd)
        k = self.wk(x).view(B, T, self.kv_heads, self.hd)
        v = self.wv(x).view(B, T, self.kv_heads, self.hd)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        # attn_bias : (B, 1, T, T) additif, -inf sur les positions interdites (futur + padding)
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_bias,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=(attn_bias is None),
        )
        return self.wo(out.transpose(1, 2).reshape(B, T, self.n_heads * self.hd))


class SwiGLU(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


class Block(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, attn_bias=None):
        x = x + self.resid_dropout(self.attn(self.attn_norm(x), cos, sin, attn_bias))
        x = x + self.resid_dropout(self.ffn(self.ffn_norm(x)))
        return x


class MiniLLM(nn.Module):
    # Mêmes noms de paramètres que inference.py -> state_dict 100% compatible.
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        self.refresh_rope()

    def refresh_rope(self):
        dev = self.tok_emb.weight.device
        cos, sin = build_rope_tables(self.cfg.head_dim, self.cfg.max_seq_len, self.cfg.rope_theta, dev)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, idx, attention_mask: Optional[torch.Tensor] = None):
        # idx: (B,T) ids. attention_mask: (B,T) 1=réel, 0=padding. Retourne les logits (B,T,V).
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]

        attn_bias = None
        if attention_mask is not None and (attention_mask == 0).any():
            causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=idx.device))
            key_ok = attention_mask.bool()[:, None, None, :]           # (B,1,1,T)
            allowed = causal[None, None, :, :] & key_ok                # (B,1,T,T)
            attn_bias = torch.zeros(B, 1, T, T, dtype=x.dtype, device=idx.device)
            attn_bias.masked_fill_(~allowed, float("-inf"))

        for block in self.blocks:
            x = block(x, cos, sin, attn_bias)
        x = self.norm_f(x)
        return self.lm_head(x)


def load_minillm(ckpt_dir):
    cfg = ModelConfig.from_json(f"{ckpt_dir}/config.json")
    model = MiniLLM(cfg)
    from safetensors.torch import load_file
    state_dict = load_file(f"{ckpt_dir}/model.safetensors")
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    return model, cfg

model, cfg = load_minillm(CKPT_DIR)
n_params = sum(p.numel() for p in model.parameters())
print(f"Modèle chargé : {n_params:,} paramètres, {cfg.n_layers} couches, d_model={cfg.d_model}, ctx={cfg.max_seq_len}")


In [ ]:
# Tokenizer + petit test de génération AVANT SFT (sanity check que le checkpoint source est bon)
from tokenizers import Tokenizer

EOT, SYSTEM, USER, ASSISTANT, END = "<|endoftext|>", "<|system|>", "<|user|>", "<|assistant|>", "<|end|>"

tok = Tokenizer.from_file(f"{CKPT_DIR}/tokenizer.json")
USER_ID, ASSIST_ID, END_ID, EOT_ID = (tok.token_to_id(t) for t in (USER, ASSISTANT, END, EOT))
PAD_ID = EOT_ID  # pas de token <pad> dédié : on réutilise <|endoftext|> et on le masque dans la loss

def encode_turn(role_id, text):
    return [role_id] + tok.encode(text.strip(), add_special_tokens=False).ids + [END_ID]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

@torch.no_grad()
def quick_generate(prompt, max_new=60, temperature=0.7, top_k=40):
    ids = encode_turn(USER_ID, prompt) + [ASSIST_ID]
    x = torch.tensor([ids], device=device)
    for _ in range(max_new):
        logits = model(x)[0, -1]
        logits = logits / temperature
        if top_k:
            kth = torch.topk(logits, top_k).values[-1]
            logits[logits < kth] = float("-inf")
        probs = torch.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1).item()
        if nxt in (END_ID, EOT_ID):
            break
        x = torch.cat([x, torch.tensor([[nxt]], device=device)], dim=1)
    out_ids = x[0, len(ids):].tolist()
    return tok.decode(out_ids, skip_special_tokens=True)

print(quick_generate("Qui es-tu ?"))


## Plan de données

Objectif : remonter la part de contenu **ancré sur le pré-entraînement** (QA extractive sur
Wikipédia FR) et **diversifier les formes** (extractif, instructions courtes, multi-tours,
conversationnel), tout en plafonnant French Alpaca pour qu'il ne domine plus le mélange.

| bloc | sources | rôle | part cible |
|---|---|---|---|
| **QA ancrée Wikipédia FR** | FQuAD, PIAF, SQuAD-FR | rappel factuel sur le corpus déjà vu au pré-entraînement | **25 %** |
| **Instructions généralistes (plafonnées)** | French Alpaca (filtré, échantillonné) | format instruction/réponse large | **20 %** (contre 62 % en v2.2) |
| **Mélange curé FR** | angeluriot/french_instruct (Dolly-FR, ShareGPT-FR, OASST-FR, WikiHow-FR, ENS Sup) | diversité de styles, dédupliqué contre les autres sources | **15 %** |
| **Multi-tours** | MaziyarPanahi/french_instruct_sharegpt, vonewman/french-instruction-dataset | dialogue à plusieurs tours | **10 %** |
| **Instructions synthétiques diversifiées** | Magpie-FR, Evol-Instruct-FR | formulations variées, complexité progressive | **10 %** |
| **Conversationnel naturel** | ultrafrench (UltraChat FR) | ton conversationnel, questions ouvertes | **6 %** |
| **Synthétique ciblé (code)** | généré ci-dessous : faits, capitales, contraires/synonymes, calendrier, calcul, logique | formats précis, vocabulaire de contrôle | **10 %** |
| **Persona** | `personnalite.jsonl` (le tien), répété ×3 (pas ×10) | identité sans déborder sur d'autres sujets | **4 %** |

Remarques :
- **FQuAD/PIAF/SQuAD-FR sont en questions-réponses extractives** (la réponse est un segment du
  contexte) : on les reformate en `<|user|>{contexte}\n\nQuestion : {question}<|end|><|assistant|>{réponse}<|end|>`
  pour forcer le modèle à *lire et répondre*, pas seulement réciter — c'est un format proche de
  ce qu'il voit déjà en pré-entraînement (texte encyclopédique), donc plus facile à généraliser.
- Les ids des datasets ci-dessous reprennent ceux que tu utilises déjà (vérifie `MaziyarPanahi`,
  `vonewman`, Magpie-FR, Evol-Instruct-FR sur ton compte HF si l'un d'eux a été renommé/supprimé —
  le code ci-dessous échoue proprement et t'indique la source en cause plutôt que de planter en silence).


In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
import re, hashlib, unicodedata, random

random.seed(1337)

SOURCES = {
    # (repo_id, config/split) -> rôle, chargés plus bas avec gestion d'erreur individuelle
    "fquad":    dict(repo="lincoln/newsquadfr", fallback="FQuAD", kind="extractive"),
    "piaf":     dict(repo="etalab-ia/piaf", kind="extractive"),
    "squad_fr": dict(repo="qwant/squad_fr", kind="extractive"),
    "french_alpaca": dict(repo="jpacifico/French-Alpaca-dataset-Instruct-110K", kind="instruct"),
    "french_instruct": dict(repo="angeluriot/french_instruct", kind="mixed"),
    "sharegpt_fr": dict(repo="MaziyarPanahi/french_instruct_sharegpt", kind="multiturn"),
    "vonewman": dict(repo="vonewman/french-instruction-dataset", kind="instruct"),
    "magpie_fr": dict(repo="jpacifico/Magpie-FR-37K", kind="instruct"),
    "evol_fr": dict(repo="jpacifico/Evol-Instruct-French-1k", kind="instruct"),
    "ultrafrench": dict(repo="AIffl/ultrafrench", kind="multiturn"),
}

def try_load(name, repo, **kw):
    try:
        ds = load_dataset(repo, split="train", token=hf_token)
        print(f"[OK] {name:16s} <- {repo:45s} {len(ds):>7,} exemples")
        return ds
    except Exception as e:
        print(f"[ABSENT] {name:16s} <- {repo:45s} ({e.__class__.__name__}: vérifie l'id exact sur HF)")
        return None

raw = {name: try_load(name, **spec) for name, spec in SOURCES.items()}
raw = {k: v for k, v in raw.items() if v is not None}


### Normalisation vers un schéma commun

Tout est ramené à `{"messages": [{"role": "user"|"assistant", "content": str}], "source": str}`.
C'est le même schéma pour une QA extractive à un tour que pour un ShareGPT à cinq tours : le
code de tokenisation / masquage de loss plus bas n'a besoin de rien connaître de la source.


In [ ]:
def norm_extractive(ds, source_name, context_key="context", question_key="question", answer_key="answers"):
    rows = []
    for ex in ds:
        ctx, q = ex.get(context_key), ex.get(question_key)
        ans = ex.get(answer_key)
        if isinstance(ans, dict):
            texts = ans.get("text") or ans.get("texts") or []
        elif isinstance(ans, list):
            texts = ans
        else:
            texts = [ans] if ans else []
        if not ctx or not q or not texts or not texts[0]:
            continue
        a = str(texts[0]).strip()
        if not (1 <= len(a) <= 200):
            continue
        user = f"{ctx.strip()}\n\nQuestion : {q.strip()}"
        rows.append({"messages": [{"role": "user", "content": user},
                                   {"role": "assistant", "content": a}],
                     "source": source_name})
    return rows

def norm_alpaca_like(ds, source_name, instr_key="instruction", input_key="input", output_key="output"):
    rows = []
    for ex in ds:
        instr = (ex.get(instr_key) or "").strip()
        inp = (ex.get(input_key) or "").strip()
        out = (ex.get(output_key) or ex.get("response") or ex.get("answer") or "").strip()
        if not instr or not out:
            continue
        user = f"{instr}\n\n{inp}".strip() if inp else instr
        rows.append({"messages": [{"role": "user", "content": user},
                                   {"role": "assistant", "content": out}],
                     "source": source_name})
    return rows

def norm_conversations(ds, source_name, key="conversations"):
    # Pour ShareGPT-like / angeluriot / ultrafrench : liste de tours {from/role, value/content}.
    role_map = {"human": "user", "user": "user", "gpt": "assistant", "assistant": "assistant",
                "system": "system"}
    rows = []
    for ex in ds:
        turns = ex.get(key) or ex.get("messages") or ex.get("conversation")
        if not turns:
            # formats à colonnes plates (prompt/response) -> fallback alpaca-like
            if ex.get("prompt") and (ex.get("response") or ex.get("completion")):
                rows.append({"messages": [
                    {"role": "user", "content": str(ex["prompt"]).strip()},
                    {"role": "assistant", "content": str(ex.get("response") or ex.get("completion")).strip()}],
                    "source": source_name})
            continue
        msgs = []
        for t in turns:
            role = role_map.get((t.get("from") or t.get("role") or "").lower())
            content = (t.get("value") or t.get("content") or "").strip()
            if role and content:
                msgs.append({"role": role, "content": content})
        if len(msgs) >= 2 and msgs[0]["role"] == "user":
            rows.append({"messages": msgs, "source": source_name})
    return rows

normalized = []
if "fquad" in raw:     normalized += norm_extractive(raw["fquad"], "fquad")
if "piaf" in raw:      normalized += norm_extractive(raw["piaf"], "piaf")
if "squad_fr" in raw:  normalized += norm_extractive(raw["squad_fr"], "squad_fr")
if "french_alpaca" in raw: normalized += norm_alpaca_like(raw["french_alpaca"], "french_alpaca")
if "french_instruct" in raw: normalized += norm_conversations(raw["french_instruct"], "french_instruct")
if "sharegpt_fr" in raw: normalized += norm_conversations(raw["sharegpt_fr"], "sharegpt_fr")
if "vonewman" in raw:  normalized += norm_alpaca_like(raw["vonewman"], "vonewman")
if "magpie_fr" in raw: normalized += norm_conversations(raw["magpie_fr"], "magpie_fr")
if "evol_fr" in raw:   normalized += norm_alpaca_like(raw["evol_fr"], "evol_fr")
if "ultrafrench" in raw: normalized += norm_conversations(raw["ultrafrench"], "ultrafrench")

print(f"Total brut normalisé : {len(normalized):,} exemples, sur {len(set(r['source'] for r in normalized))} sources")


### Filtres qualité

Mêmes filtres que ceux décrits dans la fiche v2.2 (refus génériques, boucles de répétition,
réponse qui ne fait que reprendre la question), appliqués ici à **toutes** les sources externes
(en v2.2 ils n'étaient appliqués qu'à une partie) — plus un filtre de langue (le modèle n'a vu
que du français au pré-entraînement ; des traductions bruitées en anglais/mélangées nuisent plus
qu'elles n'aident à cette échelle) et un filtre de longueur adapté au contexte de 512 tokens.


In [ ]:
REFUS_PATTERNS = [
    r"je suis d[ée]sol[ée]", r"je ne peux pas", r"en tant qu'ia", r"en tant que mod[èe]le",
    r"je n'ai pas la capacit[ée]", r"i cannot", r"i'm sorry", r"as an ai",
]
REFUS_RE = re.compile("|".join(REFUS_PATTERNS), re.IGNORECASE)

def normalize_text(s):
    s = unicodedata.normalize("NFC", s).strip().lower()
    return re.sub(r"\s+", " ", s)

def has_repetition_loop(text, n=4, min_repeats=3):
    words = text.split()
    if len(words) < n * min_repeats:
        return False
    for i in range(len(words) - n * min_repeats + 1):
        chunk = words[i:i+n]
        if all(words[i+k*n:i+(k+1)*n] == chunk for k in range(min_repeats)):
            return True
    return False

def is_mostly_french(text, sample_chars=300):
    # heuristique légère sans dépendance externe : part de caractères latins + mots-outils FR
    t = text[:sample_chars].lower()
    if not t:
        return False
    fr_markers = sum(t.count(w) for w in (" le ", " la ", " les ", " de ", " et ", " est ", " une ", " un ", " que ", " qui "))
    ascii_letters = sum(c.isalpha() for c in t)
    return ascii_letters > 0 and (fr_markers >= 1 or len(t) < 40)

def passes_quality(messages, max_chars=1800):
    user = messages[0]["content"]
    for m in messages:
        c = m["content"]
        if not (1 <= len(c) <= max_chars):
            return False
        if m["role"] == "assistant":
            if REFUS_RE.search(c):
                return False
            if has_repetition_loop(c):
                return False
            if normalize_text(c) == normalize_text(user) :
                return False
            if not is_mostly_french(c):
                return False
    return True

filtered = [r for r in normalized if passes_quality(r["messages"])]
print(f"Après filtre qualité : {len(filtered):,} / {len(normalized):,} conservés "
      f"({100*len(filtered)/max(1,len(normalized)):.1f} %)")

from collections import Counter
print(Counter(r["source"] for r in filtered))


In [ ]:
# Déduplication exacte (hash du premier tour utilisateur normalisé) — évite les quasi-doublons
# entre sources qui réutilisent les mêmes bases (Dolly/Alpaca traduits plusieurs fois, etc.)
seen = set()
deduped = []
for r in filtered:
    key = hashlib.sha1(normalize_text(r["messages"][0]["content"]).encode()).hexdigest()
    if key in seen:
        continue
    seen.add(key)
    deduped.append(r)

print(f"Après déduplication : {len(deduped):,} / {len(filtered):,} conservés")


### Données synthétiques ciblées (code)

On régénère ce bloc plutôt que de réutiliser tel quel celui de v2.2, pour deux raisons :
1. éviter que les questions d'évaluation (`test_connaissances.md`, `MiniLlmQA.md`) se
   retrouvent **mot pour mot** dans l'entraînement (c'était le cas pour 28/337 questions en v2.2) ;
2. élargir la couverture d'entités (plus de capitales, plus d'antonymes/synonymes) pour que le
   format soit appris sur un échantillon, et que le test porte sur des entités **non vues**.

Remplace/complète `CAPITALS`, `ANTONYMS`, etc. par tes propres listes si tu veux garder un
contrôle total sur ce qui est appris par cœur vs retenu par généralisation.


In [ ]:
CAPITALS = {
    "France": "Paris", "Allemagne": "Berlin", "Italie": "Rome", "Espagne": "Madrid",
    "Portugal": "Lisbonne", "Belgique": "Bruxelles", "Suisse": "Berne", "Cameroun": "Yaoundé",
    "Sénégal": "Dakar", "Côte d'Ivoire": "Yamoussoukro", "Nigeria": "Abuja", "Kenya": "Nairobi",
    "Maroc": "Rabat", "Algérie": "Alger", "Tunisie": "Tunis", "Égypte": "Le Caire",
    "Canada": "Ottawa", "États-Unis": "Washington", "Brésil": "Brasília", "Japon": "Tokyo",
    "Chine": "Pékin", "Inde": "New Delhi", "Russie": "Moscou", "Royaume-Uni": "Londres",
    # garde-en 15 à 20 hors de cette liste pour l'évaluation (ne pas tout mettre dans le SFT)
}
ANTONYMS = [("grand", "petit"), ("chaud", "froid"), ("rapide", "lent"), ("fort", "faible"),
            ("jour", "nuit"), ("facile", "difficile"), ("beau", "laid"), ("riche", "pauvre"),
            ("ouvert", "fermé"), ("propre", "sale"), ("plein", "vide"), ("calme", "agité")]
SYNONYMS = [("content", "heureux"), ("rapide", "véloce"), ("commencer", "débuter"),
            ("terminer", "finir"), ("grand", "immense"), ("beau", "magnifique")]

def gen_capitals(n_eval_holdout=6):
    items = list(CAPITALS.items())
    random.shuffle(items)
    train_items = items[n_eval_holdout:]  # garde les premiers de côté pour une éval externe
    rows = []
    for pays, cap in train_items:
        rows.append({"messages": [
            {"role": "user", "content": f"Quelle est la capitale de {pays} ?"},
            {"role": "assistant", "content": f"La capitale de {pays} est {cap}."}],
            "source": "synth_capitales"})
    return rows

def gen_antonyms():
    rows = []
    for a, b in ANTONYMS:
        rows.append({"messages": [{"role": "user", "content": f"Quel est le contraire de {a} ?"},
                                   {"role": "assistant", "content": f"Le contraire de {a} est {b}."}],
                     "source": "synth_contraires"})
        rows.append({"messages": [{"role": "user", "content": f"Quel est le contraire de {b} ?"},
                                   {"role": "assistant", "content": f"Le contraire de {b} est {a}."}],
                     "source": "synth_contraires"})
    return rows

def gen_synonyms():
    rows = []
    for a, b in SYNONYMS:
        rows.append({"messages": [{"role": "user", "content": f"Donne un synonyme de {a}."},
                                   {"role": "assistant", "content": f"Un synonyme de {a} est {b}."}],
                     "source": "synth_synonymes"})
    return rows

def gen_arithmetic(n=400, max_val=50):
    ops = [("+", lambda a,b: a+b, "plus"), ("-", lambda a,b: a-b, "moins"), ("*", lambda a,b: a*b, "fois")]
    rows = []
    for _ in range(n):
        a, b = random.randint(1, max_val), random.randint(1, max_val)
        sym, fn, mot = random.choice(ops)
        if sym == "-" and b > a:
            a, b = b, a
        res = fn(a, b)
        rows.append({"messages": [{"role": "user", "content": f"Combien font {a} {mot} {b} ?"},
                                   {"role": "assistant", "content": f"{a} {mot} {b} font {res}."}],
                     "source": "synth_calcul"})
    return rows

synthetic = gen_capitals() + gen_antonyms() + gen_synonyms() + gen_arithmetic()
print(f"Synthétique : {len(synthetic):,} exemples")


### Persona

Charge ton `personnalite.jsonl` existant (même format `{"messages": [...]}` que le reste). On le
répète **×3** au lieu de ×10 : en v2.2, 5,8 % du SFT était de la persona répétée ×10, ce qui
corrèle avec le débordement observé (« Git est un langage de programmation développé par
Git.com », « SQL... développé par DevLab »).


In [ ]:
import os

PERSONA_PATH = "/content/personnalite.jsonl"  # dépose le fichier dans Colab avant d'exécuter
PERSONA_REPEAT = 3

persona = []
if os.path.exists(PERSONA_PATH):
    with open(PERSONA_PATH, encoding="utf-8") as f:
        base_persona = [json.loads(l) for l in f if l.strip()]
    for r in base_persona:
        r.setdefault("source", "persona")
    persona = base_persona * PERSONA_REPEAT
    print(f"Persona : {len(base_persona)} exemples de base, ×{PERSONA_REPEAT} = {len(persona)}")
else:
    print(f"Persona non trouvée à {PERSONA_PATH} — dépose le fichier puis relance cette cellule.")


### Composition finale par quotas

On échantillonne chaque source pour viser les proportions de la section « Plan de données »,
sur un total cible (ajustable). Les sources plus petites que leur quota sont prises en entier
(pas de sur-échantillonnage artificiel, sauf la persona déjà gérée par répétition ci-dessus).


In [ ]:
TARGET_TOTAL = 45000

TARGETS = {
    # (sources regroupées) -> fraction cible
    frozenset({"fquad", "piaf", "squad_fr"}): 0.25,
    frozenset({"french_alpaca"}): 0.20,
    frozenset({"french_instruct"}): 0.15,
    frozenset({"sharegpt_fr", "vonewman"}): 0.10,
    frozenset({"magpie_fr", "evol_fr"}): 0.10,
    frozenset({"ultrafrench"}): 0.06,
    frozenset({"synth_capitales", "synth_contraires", "synth_synonymes", "synth_calcul"}): 0.10,
    frozenset({"persona"}): 0.04,
}

by_source = {}
for r in deduped + synthetic + persona:
    by_source.setdefault(r["source"], []).append(r)

final = []
for group, frac in TARGETS.items():
    pool = [r for s in group for r in by_source.get(s, [])]
    random.shuffle(pool)
    quota = int(TARGET_TOTAL * frac)
    take = pool if s == "persona" else pool[:quota] if len(pool) > quota else pool
    # la persona n'est pas sous-échantillonnée : on la garde entière (déjà répétée ×3 ci-dessus)
    if group == frozenset({"persona"}):
        take = pool
    final += take
    print(f"{sorted(group)!s:70s} quota={quota:>6,}  dispo={len(pool):>6,}  pris={len(take):>6,}")

random.shuffle(final)
print(f"\nTotal final : {len(final):,} exemples")
print(Counter(r["source"] for r in final))


### Split train / validation par hachage

Même logique que le split du pré-entraînement (hachage du texte, pas un split aléatoire par
index) : stable, reproductible, et évite qu'une quasi-duplication entre deux lignes tombe de part
et d'autre du split.


In [ ]:
def split_hash(rows, val_frac=0.02):
    train, val = [], []
    for r in rows:
        h = int(hashlib.sha1(normalize_text(r['messages'][0]['content']).encode()).hexdigest(), 16)
        (val if (h % 10000) < val_frac * 10000 else train).append(r)
    return train, val

train_rows, val_rows = split_hash(final)
print(f"train={len(train_rows):,}  val={len(val_rows):,}")


## Tokenisation, masquage de la loss, padding dynamique

Format identique à `inference.py` / au tokenizer du dépôt :
`<|user|>…<|end|><|assistant|>…<|end|>`, loss calculée **uniquement sur les tokens de réponse**
(y compris le `<|end|>` de fin de réponse, pour que le modèle apprenne à s'arrêter).


In [ ]:
SYSTEM_ID = tok.token_to_id(SYSTEM)
ROLE_ID = {"user": USER_ID, "assistant": ASSIST_ID, "system": SYSTEM_ID}
IGNORE_INDEX = -100

def encode_example(messages, max_len=cfg.max_seq_len):
    ids, labels = [], []
    for m in messages:
        turn_ids = [ROLE_ID[m["role"]]] + tok.encode(m["content"].strip(), add_special_tokens=False).ids + [END_ID]
        ids += turn_ids
        if m["role"] == "assistant":
            labels += turn_ids
        else:
            labels += [IGNORE_INDEX] * len(turn_ids)
    ids, labels = ids[:max_len], labels[:max_len]
    return ids, labels

class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.examples = [encode_example(r["messages"]) for r in rows]
        # on jette les exemples où la loss serait calculée sur rien (troncature ayant mangé la réponse)
        self.examples = [(i, l) for i, l in self.examples if any(x != IGNORE_INDEX for x in l)]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate(batch):
    maxlen = max(len(ids) for ids, _ in batch)
    B = len(batch)
    input_ids = torch.full((B, maxlen), PAD_ID, dtype=torch.long)
    labels = torch.full((B, maxlen), IGNORE_INDEX, dtype=torch.long)
    attn_mask = torch.zeros((B, maxlen), dtype=torch.long)
    for i, (ids, lab) in enumerate(batch):
        L = len(ids)
        input_ids[i, :L] = torch.tensor(ids)
        labels[i, :L] = torch.tensor(lab)
        attn_mask[i, :L] = 1
    return input_ids, labels, attn_mask

train_ds, val_ds = SFTDataset(train_rows), SFTDataset(val_rows)
print(f"train={len(train_ds):,} exemples tokenisés  val={len(val_ds):,}")

BATCH_SIZE = 16
GRAD_ACCUM = 2
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)


## Jeu de sondage factuel (tenu à l'écart du SFT)

Un petit sous-ensemble, **distinct** des sources d'entraînement (capitales mises de côté plus
haut + quelques faits fixes), pour choisir le checkpoint sur autre chose que `val_loss` — comme
tu l'as déjà constaté, la `val_loss` seule peut désigner un checkpoint qui a perdu la persona et
les connaissances. Complète cette liste avec ton propre `test_connaissances.md` pour un signal
plus fiable.


In [ ]:
held_out_capitals = dict(list(CAPITALS.items())[:6])  # ceux mis de côté dans gen_capitals()
PROBE_SET = [(f"Quelle est la capitale de {p} ?", c) for p, c in held_out_capitals.items()] + [
    ("Qui es-tu ?", "miniLLM"),
    ("Quelle est la planète la plus proche du Soleil ?", "Mercure"),
    ("Quel est le plus long fleuve d'Afrique ?", "Nil"),
]

@torch.no_grad()
def run_probe(model, verbose=False):
    model.eval()
    correct = 0
    for q, expected in PROBE_SET:
        ans = quick_generate(q, max_new=40, temperature=0.0)
        ok = expected.lower() in ans.lower()
        correct += ok
        if verbose:
            print(f"{'✓' if ok else '✗'} Q: {q}\n   R: {ans}")
    model.train()
    return correct / len(PROBE_SET)


## Boucle d'entraînement

Mêmes hyperparamètres de départ que v2.2 (qui a fonctionné mieux que les LR plus bas testés sans
succès), appliqués au mélange de données révisé : AdamW, warmup 100 it, cosine decay, 2 époques,
dropout 0.1, clipping à 1.0, fp16. Checkpoint sauvegardé à chaque éval ; sélection finale sur le
score du jeu de sondage (pas uniquement `val_loss`).


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

EPOCHS = 2
LR = 1e-4
MIN_LR = 1e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
EVAL_EVERY = 250
SAVE_DIR = "/content/sft_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

model.cfg.dropout = 0.1
for m in model.modules():
    if isinstance(m, (nn.Dropout,)):
        m.p = 0.1

total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
optim = AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    prog = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    cos = 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
    return (MIN_LR / LR) + (1 - MIN_LR / LR) * cos

sched = LambdaLR(optim, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

@torch.no_grad()
def eval_loss(loader, max_batches=40):
    model.eval()
    tot, n = 0.0, 0
    for i, (ids, labels, mask) in enumerate(loader):
        if i >= max_batches:
            break
        ids, labels, mask = ids.to(device), labels.to(device), mask.to(device)
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu", dtype=torch.float16, enabled=(device=="cuda")):
            logits = model(ids, attention_mask=mask)
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                                    labels[:, 1:].reshape(-1), ignore_index=IGNORE_INDEX)
        tot += loss.item(); n += 1
    model.train()
    return tot / max(1, n)

history = []
best_probe, best_step = -1.0, -1
step = 0
model.train()
for epoch in range(EPOCHS):
    optim.zero_grad()
    for i, (ids, labels, mask) in enumerate(train_loader):
        ids, labels, mask = ids.to(device), labels.to(device), mask.to(device)
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu", dtype=torch.float16, enabled=(device=="cuda")):
            logits = model(ids, attention_mask=mask)
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                                    labels[:, 1:].reshape(-1), ignore_index=IGNORE_INDEX)
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()

        if (i + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optim)
            scaler.update()
            optim.zero_grad()
            sched.step()
            step += 1

            if step % EVAL_EVERY == 0:
                vl = eval_loss(val_loader)
                probe = run_probe(model)
                history.append({"step": step, "val_loss": vl, "probe_acc": probe, "lr": sched.get_last_lr()[0]})
                print(f"step {step:>5} | epoch {epoch+1} | val_loss {vl:.4f} | probe {probe:.1%} | lr {sched.get_last_lr()[0]:.2e}")
                ckpt_path = f"{SAVE_DIR}/step_{step}.pt"
                torch.save(model.state_dict(), ckpt_path)
                if probe >= best_probe:  # en cas d'égalité on préfère le plus récent (plus de signal vu)
                    best_probe, best_step = probe, step

print(f"\nMeilleur checkpoint sur le jeu de sondage : step {best_step} (probe={best_probe:.1%})")


In [ ]:
# Toujours sauvegarder aussi le DERNIER checkpoint (comme en v2.2) pour comparer les deux
torch.save(model.state_dict(), f"{SAVE_DIR}/last.pt")

import pandas as pd
hist_df = pd.DataFrame(history)
display(hist_df)


## Évaluation qualitative avant publication

Compare le checkpoint `best` (meilleur score de sondage) et `last` (dernière itération, comme en
v2.2) sur quelques questions ouvertes, **avant** de décider lequel publier — c'est exactement la
vérification qui t'avait fait préférer `last` à `best val_loss` en v2.2 : refais-la ici plutôt que
de supposer que le score du sondage suffit seul.


In [ ]:
QUESTIONS_LIBRES = [
    "Qui es-tu ?",
    "Qu'est-ce que la photosynthèse ?",
    "Quelle est la capitale du Sénégal ?",   # à choisir hors de CAPITALS si tu veux un vrai test
    "Qu'est-ce que Git ?",
    "Corrige cette phrase : Les enfant joue dehors.",
    "Invente une devinette simple.",
]

def load_ckpt(path):
    sd = torch.load(path, map_location=device)
    model.load_state_dict(sd)
    model.eval()

for tag, path in [("best", f"{SAVE_DIR}/step_{best_step}.pt"), ("last", f"{SAVE_DIR}/last.pt")]:
    print(f"\n===== {tag} ({path}) =====")
    load_ckpt(path)
    for q in QUESTIONS_LIBRES:
        print(f"Q: {q}\nR: {quick_generate(q, max_new=60, temperature=0.0)}\n")


## Publication

Choisis `CHOSEN_PATH` après avoir lu la comparaison ci-dessus (pas automatiquement `best`),
reconstruis un `model.safetensors` propre, et pousse vers `DEST_REPO` avec le `config.json` /
`tokenizer.json` / `generation_config.json` inchangés (même architecture, même vocabulaire).


In [ ]:
CHOSEN_PATH = f"{SAVE_DIR}/last.pt"  # ajuste selon la comparaison qualitative ci-dessus

load_ckpt(CHOSEN_PATH)

from safetensors.torch import save_file
OUT_DIR = "/content/final_sft"
os.makedirs(OUT_DIR, exist_ok=True)

sd = {k: v.contiguous() for k, v in model.state_dict().items()}
save_file(sd, f"{OUT_DIR}/model.safetensors")
for fname in ["config.json", "generation_config.json", "tokenizer.json"]:
    shutil.copy(f"{CKPT_DIR}/{fname}", f"{OUT_DIR}/{fname}")

print(os.listdir(OUT_DIR))

if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    api.create_repo(DEST_REPO, exist_ok=True)
    api.upload_folder(folder_path=OUT_DIR, repo_id=DEST_REPO)
    print(f"Publié sur https://huggingface.co/{DEST_REPO}")
else:
    print("PUSH_TO_HUB=False — checkpoint prêt localement dans", OUT_DIR)


## Pour la suite (v3 / pré-entraînement)

- Le gain le plus net viendra probablement d'une **phase finale de pré-entraînement mêlant du
  texte encyclopédique à du format Q/R** (sur le modèle de ce que fait ce SFT, mais à l'échelle
  du pré-entraînement) plutôt que d'empiler encore des sources de SFT génériques — tu avais déjà
  noté cette piste dans tes notes de projet.
- Si tu listes toi-même quelques milliers de paires question/réponse à partir d'extraits de ton
  corpus Wikipédia FR (même mécanique que PIAF/FQuAD, sur **tes propres documents** plutôt que
  sur le Wikipédia générique de ces datasets), le recouvrement avec ce que le modèle a réellement
  mémorisé serait meilleur qu'avec des QA externes.
- Garde un œil sur la taille du jeu de sondage : 10 questions comme ci-dessus donnent un signal
  bruité (±10 points) — branche ton `test_connaissances.md` complet (204 questions) dès que
  possible pour un signal comparable à celui de la fiche du modèle.
